In [2]:
import logging
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
import yfinance as yf

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("notebook_tester")

print("Libraries imported and logger initialized.")

Libraries imported and logger initialized.


In [3]:
def format_tickers(ticker: str) -> str:
    ticker = ticker.strip().upper()
    if not ticker.endswith(".NS"):
        ticker += ".NS"
    return ticker


# Test formatting variations
print("Testing format_tickers:")
print("  'infy'    ->", format_tickers("infy"))
print("  'TCS.NS'  ->", format_tickers("TCS.NS"))
print("  '  wipro ' ->", format_tickers("  wipro "))

Testing format_tickers:
  'infy'    -> INFY.NS
  'TCS.NS'  -> TCS.NS
  '  wipro ' -> WIPRO.NS


In [4]:
def pick_OHLCV(
    ticker: str, start: str, end: str, interval: str = "1d"
) -> pd.DataFrame:
    index = format_tickers(ticker)
    df = yf.download(
        index,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=True,
        progress=False,
    )

    if df.empty:
        raise ValueError(f"No data returned for {index}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df.columns = [c.lower() for c in df.columns]
    df = df[["date", "open", "high", "low", "close", "volume"]]
    df["ticker"] = index
    return df


# Test real download for a single stock
single_stock_raw = pick_OHLCV("INFY", "2023-01-01", "2023-06-01")
print(f"Downloaded INFY shape: {single_stock_raw.shape}")
single_stock_raw.head(3)

Downloaded INFY shape: (101, 7)


,date,open,high,low,close,volume,ticker
0,2023-01-02,1359.120274,1370.341544,1350.188186,1368.097290,3801140,INFY.NS
1,2023-01-03,1364.371672,1375.862296,1360.107590,1366.795532,4551130,INFY.NS
2,2023-01-04,1365.404262,1365.404262,1338.562960,1341.884521,7059590,INFY.NS


In [5]:
def ohlcv_validation(
    df: pd.DataFrame,
    ticker: str,
    max_row_missing_pct: float = 2.0,
    max_consecutive_gap: int = 3,
) -> dict:
    row_missing_pct = round(df.isnull().any(axis=1).mean() * 100, 2)
    col_missing_pct = (df.isnull().mean() * 100).round(2).to_dict()

    is_null_close = df["close"].isnull()
    if is_null_close.any():
        max_gap = int(
            is_null_close.groupby((~is_null_close).cumsum()).sum().max()
        )
    else:
        max_gap = 0

    report = {
        "ticker": ticker,
        "rows": len(df),
        "row_missing_pct": row_missing_pct,
        "col_missing_pct": col_missing_pct,
        "max_consecutive_gap_days": max_gap,
        "zero_volume_days": int((df["volume"] == 0).sum()),
        "duplicate_dates": int(df["date"].duplicated().sum()),
        "sorted": bool(df["date"].is_monotonic_increasing),
    }

    report["ok"] = (
        row_missing_pct <= max_row_missing_pct
        and max_gap <= max_consecutive_gap
        and report["duplicate_dates"] == 0
    )
    return report


# Test validation report on downloaded data
validation_output = ohlcv_validation(single_stock_raw, "INFY.NS")
print("Validation Report:")
for k, v in validation_output.items():
    print(f"  {k}: {v}")

Validation Report:
  ticker: INFY.NS
  rows: 101
  row_missing_pct: 0.0
  col_missing_pct: {'date': 0.0, 'open': 0.0, 'high': 0.0, 'low': 0.0, 'close': 0.0, 'volume': 0.0, 'ticker': 0.0}
  max_consecutive_gap_days: 0
  zero_volume_days: 0
  duplicate_dates: 0
  sorted: True
  ok: True


In [6]:
def fetch_grp(
    tickers: list[str],
    start: str,
    end: str,
    min_success_rate: float = 0.5,
) -> tuple[pd.DataFrame, list[dict]]:
    frames = []
    reports = []

    for t in tickers:
        try:
            df = pick_OHLCV(t, start, end)
            report = ohlcv_validation(df, t)
            reports.append(report)

            if not report["ok"]:
                logger.warning(f"{t}: quality issue -> {report}")
            frames.append(df)

        except ValueError as e:
            logger.error(f"{t}: fetch failed -> {e}")
            reports.append({"ticker": t, "ok": False, "error": str(e)})

    if not frames:
        failed = [r["ticker"] for r in reports]
        raise RuntimeError(f"All {len(tickers)} tickers failed: {failed}")

    success_rate = len(frames) / len(tickers)
    if success_rate < min_success_rate:
        raise RuntimeError(
            f"Success rate {success_rate:.0%} < {min_success_rate:.0%}"
        )

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["ticker", "date"]).reset_index(drop=True)
    return combined, reports


# Test multi-ticker batch download
test_portfolio = ["INFY", "TCS", "WIPRO"]
raw_combined_df, audit_reports = fetch_grp(
    test_portfolio, "2023-01-01", "2023-06-01"
)
print(f"Batch download complete. Shape: {raw_combined_df.shape}")
print("Tickers fetched:", raw_combined_df["ticker"].unique().tolist())

Batch download complete. Shape: (303, 7)
Tickers fetched: ['INFY.NS', 'TCS.NS', 'WIPRO.NS']


In [7]:
def drop_badtickers(
    combined: pd.DataFrame, reports: list[dict]
) -> pd.DataFrame:
    tickers_good = [r["ticker"] for r in reports if r.get("ok", False)]
    tickers_bad = [r["ticker"] for r in reports if not r.get("ok", False)]
    if tickers_bad:
        logger.warning(
            f"Dropping tickers that failed quality checks: {tickers_bad}"
        )

    if not tickers_good:
        raise RuntimeError(
            "No tickers passed quality checks - nothing to preprocess."
        )
    good_formatted = [
        t if t.endswith(".NS") else t.upper() + ".NS" for t in tickers_good
    ]

    filtered = combined[combined["ticker"].isin(good_formatted)].copy()
    return filtered


# Test ticker filtering
clean_filtered_df = drop_badtickers(raw_combined_df, audit_reports)
print(f"Filtered DataFrame rows: {len(clean_filtered_df)}")

Filtered DataFrame rows: 303


In [8]:
def small_gap_skip(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["ticker", "date"]).copy()
    value_cols = ["open", "high", "low", "close", "volume"]
    df[value_cols] = df.groupby("ticker")[value_cols].ffill()

    remaining_na = df.isnull().sum().sum()
    if remaining_na > 0:
        logger.warning(
            f"{remaining_na} values still NaN after ffill - dropping rows"
        )
        df = df.dropna()

    return df.reset_index(drop=True)


# Test forward imputation
clean_no_gaps_df = small_gap_skip(clean_filtered_df)
print(f"Total NaNs after small_gap_skip: {clean_no_gaps_df.isnull().sum().sum()}")

Total NaNs after small_gap_skip: 0


In [9]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period, min_periods=period, adjust=False
    ).mean()
    avg_loss = loss.ewm(
        alpha=1 / period, min_periods=period, adjust=False
    ).mean()

    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))


# Test on real close series
sample_close = single_stock_raw["close"]
rsi_series = compute_rsi(sample_close, period=14)
print(f"RSI min: {rsi_series.min():.2f} | max: {rsi_series.max():.2f}")
rsi_series.tail(3)

RSI min: 22.33 | max: 67.70


98     57.504459
99     60.075242
100    57.861429
Name: close, dtype: float64

In [10]:
def compute_macd(
    series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9
) -> tuple[pd.Series, pd.Series, pd.Series]:
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram


# Test MACD calculation
macd, macd_sig, macd_h = compute_macd(sample_close)
pd.DataFrame(
    {"MACD": macd, "Signal": macd_sig, "Histogram": macd_h}
).tail(3)

,MACD,Signal,Histogram
98,-1.487221,-12.098261,10.611040
99,1.076235,-9.463362,10.539596
100,2.674946,-7.035700,9.710646


In [11]:
def compute_bollinger_bands(
    series: pd.Series, period: int = 20, num_std: float = 2.0
) -> tuple[pd.Series, pd.Series, pd.Series]:
    sma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    upper = sma + (num_std * std)
    lower = sma - (num_std * std)
    return upper, sma, lower


# Test Bollinger calculation
bb_up, bb_mid, bb_low = compute_bollinger_bands(sample_close)
pd.DataFrame(
    {"Upper": bb_up, "Middle": bb_mid, "Lower": bb_low, "Close": sample_close}
).tail(3)

,Upper,Middle,Lower,Close
98,1184.497455,1144.221753,1103.946051,1179.938965
99,1191.179856,1146.304419,1101.428982,1188.422241
100,1196.190358,1148.510522,1100.830687,1183.440186


In [12]:
def compute_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = (
        close.diff()
        .fillna(0)
        .apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
    )
    return (direction * volume).cumsum()


# Test OBV calculation
sample_vol = single_stock_raw["volume"]
obv_series = compute_obv(sample_close, sample_vol)
print("OBV last 3 values:")
print(obv_series.tail(3))

OBV last 3 values:
98    -73258133
99    -66983414
100   -84784196
dtype: int64


In [13]:
def build_indicators_for_ticker(
    df: pd.DataFrame,
    horizon_days: int = 3,
    flat_threshold_pct: float = 0.0075,
) -> pd.DataFrame:
    df = df.sort_values("date").copy()

    df["ema_9"] = df["close"].ewm(span=9, adjust=False).mean()
    df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
    df["sma_50"] = df["close"].rolling(window=50).mean()
    df["dist_ema_21"] = (df["close"] - df["ema_21"]) / df["ema_21"]
    df["dist_sma_50"] = (df["close"] - df["sma_50"]) / df["sma_50"]

    df["rsi_14"] = compute_rsi(df["close"], period=14)
    df["macd"], df["macd_signal"], df["macd_hist"] = compute_macd(df["close"])

    bb_upper, bb_mid, bb_lower = compute_bollinger_bands(df["close"])
    df["bb_width"] = (bb_upper - bb_lower) / (bb_mid + 1e-9)
    df["bb_pct"] = (df["close"] - bb_lower) / (bb_upper - bb_lower + 1e-9)

    df["vol_sma_20"] = df["volume"].rolling(window=20).mean()
    df["vol_ratio"] = df["volume"] / (df["vol_sma_20"] + 1e-9)
    df["obv"] = compute_obv(df["close"], df["volume"])
    df["obv_ema_20"] = df["obv"].ewm(span=20, adjust=False).mean()
    df["obv_divergence"] = (df["obv"] - df["obv_ema_20"]) / (
        df["obv_ema_20"].abs() + 1e-9
    )

    forward_return = (
        df["close"].shift(-horizon_days) - df["close"]
    ) / df["close"]
    df["target_return"] = forward_return

    conditions = [
        forward_return > flat_threshold_pct,
        forward_return < -flat_threshold_pct,
    ]
    choices = [1, -1]
    df["target"] = np.select(conditions, choices, default=0)

    df = df.head(-horizon_days).copy()
    df = df.dropna().reset_index(drop=True)
    return df


# Test on a single ticker frame (using 1 full year for sufficient 50-day window)
full_year_infy = pick_OHLCV("INFY", "2023-01-01", "2024-01-01")
single_feat_df = build_indicators_for_ticker(full_year_infy)
print(f"Features DataFrame shape: {single_feat_df.shape}")
single_feat_df[
    ["date", "close", "rsi_14", "dist_sma_50", "bb_pct", "target"]
].head(3)

Features DataFrame shape: (193, 25)


,date,close,rsi_14,dist_sma_50,bb_pct,target
0,2023-03-14,1274.736450,28.944967,-0.071856,0.048670,0
1,2023-03-15,1274.422241,28.899391,-0.070817,0.098755,-1
2,2023-03-16,1260.462769,26.874740,-0.079568,0.084573,-1


In [14]:
def generate_feature_dataset(
    df: pd.DataFrame,
    horizon_days: int = 3,
    flat_threshold_pct: float = 0.0075,
) -> pd.DataFrame:
    processed = []
    for ticker, group in df.groupby("ticker"):
        if len(group) < 60:
            logger.warning(
                f"Skipping {ticker}: only {len(group)} rows (needs >= 60)."
            )
            continue
        feat_group = build_indicators_for_ticker(
            group,
            horizon_days=horizon_days,
            flat_threshold_pct=flat_threshold_pct,
        )
        processed.append(feat_group)

    if not processed:
        raise ValueError("No feature frames generated.")

    out = pd.concat(processed, ignore_index=True)
    return out.sort_values(["ticker", "date"]).reset_index(drop=True)


# Test batch feature generation
# Download 1-year data so every ticker clears the 60-bar threshold
batch_raw, batch_reports = fetch_grp(
    ["INFY", "TCS"], "2023-01-01", "2024-01-01"
)
all_features_df = generate_feature_dataset(batch_raw)
print(f"Generated multi-ticker features shape: {all_features_df.shape}")
print("Class counts in target:")
print(all_features_df["target"].value_counts())

Generated multi-ticker features shape: (386, 25)
Class counts in target:
target
 1    143
 0    135
-1    108
Name: count, dtype: int64


In [15]:
def chronological_split(
    df: pd.DataFrame,
    train_frac: float = 0.70,
    val_frac: float = 0.20,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_parts, val_parts, test_parts = [], [], []

    for ticker, group in df.groupby("ticker"):
        group = group.sort_values("date").reset_index(drop=True)
        n = len(group)

        train_end = int(n * train_frac)
        val_end = train_end + int(n * val_frac)

        train_parts.append(group.iloc[:train_end])
        val_parts.append(group.iloc[train_end:val_end])
        test_parts.append(group.iloc[val_end:])

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df


# Test split
train_df, val_df, test_df = chronological_split(
    all_features_df, train_frac=0.70, val_frac=0.20
)
print(
    f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}"
)

Train rows: 270 | Val rows: 76 | Test rows: 40


In [16]:
def evaluate_predictions(
    y_true: pd.Series, y_pred: pd.Series, title: str = "Baseline"
):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\n================ {title} ================")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")
    print("Classification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["DOWN (-1)", "FLAT (0)", "UP (1)"],
            zero_division=0,
        )
    )
    return {"accuracy": acc, "macro_f1": macro_f1}


# Test evaluation reporting with dummy arrays
y_dummy_true = pd.Series([1, -1, 0, 1, 0, -1])
y_dummy_pred = pd.Series([1, -1, 0, 0, 0, -1])
test_metrics = evaluate_predictions(
    y_dummy_true, y_dummy_pred, title="Dummy Verification"
)


================ Dummy Verification ================
Accuracy : 0.8333
Macro F1 : 0.8222
Classification Report:
              precision    recall  f1-score   support

   DOWN (-1)       1.00      1.00      1.00         2
    FLAT (0)       0.67      1.00      0.80         2
      UP (1)       1.00      0.50      0.67         2

    accuracy                           0.83         6
   macro avg       0.89      0.83      0.82         6
weighted avg       0.89      0.83      0.82         6



In [17]:
def run_baselines(val_df: pd.DataFrame, lookback_days: int = 3):
    y_true = val_df["target"]

    # 1. Majority Class Baseline
    majority_class = y_true.mode()[0]
    y_pred_majority = pd.Series(majority_class, index=val_df.index)
    evaluate_predictions(
        y_true, y_pred_majority, title="Majority Class Baseline"
    )

    # 2. Momentum / Persistence Baseline
    past_return = (val_df["close"] - val_df["close"].shift(lookback_days)) / (
        val_df["close"].shift(lookback_days) + 1e-9
    )
    flat_threshold = 0.0075

    conditions = [
        past_return > flat_threshold,
        past_return < -flat_threshold,
    ]
    choices = [1, -1]
    y_pred_momentum = pd.Series(
        np.select(conditions, choices, default=0), index=val_df.index
    )

    evaluate_predictions(
        y_true, y_pred_momentum, title=f"{lookback_days}-Day Momentum Baseline"
    )


# Execute baselines directly on the validation dataset
run_baselines(val_df, lookback_days=3)


================ Majority Class Baseline ================
Accuracy : 0.3684
Macro F1 : 0.1795
Classification Report:
              precision    recall  f1-score   support

   DOWN (-1)       0.00      0.00      0.00        27
    FLAT (0)       0.00      0.00      0.00        21
      UP (1)       0.37      1.00      0.54        28

    accuracy                           0.37        76
   macro avg       0.12      0.33      0.18        76
weighted avg       0.14      0.37      0.20        76


================ 3-Day Momentum Baseline ================
Accuracy : 0.3158
Macro F1 : 0.3087
Classification Report:
              precision    recall  f1-score   support

   DOWN (-1)       0.32      0.30      0.31        27
    FLAT (0)       0.23      0.24      0.23        21
      UP (1)       0.38      0.39      0.39        28

    accuracy                           0.32        76
   macro avg       0.31      0.31      0.31        76
weighted avg       0.32      0.32      0.32        76

